# 02 - Feature Extraction (Audio + Video + Text)

This notebook demonstrates loading pre-extracted features for all three modalities.

**Supported features (from the uploaded set):**
- **Audio**: BoAW_openSMILE eGeMAPS, BoAW_openSMILE MFCC, OpenSMILE2.3.0 eGeMAPS, OpenSMILE2.3.0 MFCC
- **Video**: OpenFace2.1.0 Pose/Gaze/AUs, CNN ResNet, CNN VGG, DenseNet201, VGG16
- **Text**: Transcripts encoded with SBERT (all-MiniLM-L6-v2)

All features are loaded from the organized `data/patients/{pid}/` folders.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.boaw_extractor import BoAWExtractor
from src.features.openface_extractor import OpenFaceExtractor
from src.features.cnn_feature_loader import CNNFeatureLoader
from src.features.video_feature_extractor import VideoFeatureExtractor
from src.features.text_extractor import TextExtractor

PATIENTS_DIR = Path("../data/patients")
sample_pid = sorted([d.name for d in PATIENTS_DIR.iterdir() if d.is_dir()])[0]
print(f"Sample patient: {sample_pid}")

## 2.1 Load Audio Features

In [ ]:
boaw = BoAWExtractor()
patient_folder = PATIENTS_DIR / sample_pid / "audio_features"

# Discover files
audio_files = list(patient_folder.iterdir()) if patient_folder.exists() else []
print("Audio feature files:")
for f in audio_files:
    print(f"  {f.name}")

# Load BoAW eGeMAPS
boaw_egemaps_path = [f for f in audio_files if 'boaw' in f.name.lower() and 'egemap' in f.name.lower()]
if boaw_egemaps_path:
    boaw_egemaps = boaw.load_boaw_egemaps(boaw_egemaps_path[0])
    print(f"\nBoAW eGeMAPS shape: {boaw_egemaps.shape}")
    print(f"  Mean: {boaw_egemaps.mean():.4f}, Std: {boaw_egemaps.std():.4f}")

# Load OpenSMILE MFCC
opensmile_mfcc_path = [f for f in audio_files if 'opensmile' in f.name.lower() and 'mfcc' in f.name.lower()]
if opensmile_mfcc_path:
    opensmile_mfcc = boaw.load_opensmile_mfcc(opensmile_mfcc_path[0])
    print(f"\nOpenSMILE MFCC shape: {opensmile_mfcc.shape}")
    print(f"  Mean: {opensmile_mfcc.mean():.4f}, Std: {opensmile_mfcc.std():.4f}")

## 2.2 Load Video Features (OpenFace + CNN)

In [ ]:
video_extractor = VideoFeatureExtractor()
video_features = video_extractor.extract_from_folder(PATIENTS_DIR / sample_pid)

if video_features['openface'] is not None:
    print(f"OpenFace shape: {video_features['openface'].shape}")  # (T, 49)
    print(f"  Pose (6D):   mean={video_features['openface'][:, :6].mean():.3f}")
    print(f"  Gaze (8D):   mean={video_features['openface'][:, 6:14].mean():.3f}")
    print(f"  AUs (35D):   mean={video_features['openface'][:, 14:].mean():.3f}")

if video_features['cnn_embed'] is not None:
    print(f"\nCNN embed shape: {video_features['cnn_embed'].shape}")  # (T, D)
    print(f"  Mean: {video_features['cnn_embed'].mean():.4f}, Std: {video_features['cnn_embed'].std():.4f}")

## 2.3 Load / Encode Text Features

In [ ]:
text_extractor = TextExtractor()
transcript_path = PATIENTS_DIR / sample_pid / "transcript.csv"

if transcript_path.exists():
    text_embeddings = text_extractor.load_from_csv(transcript_path)
    print(f"Text embeddings shape: {text_embeddings.shape}")  # (N, 384)
    print(f"  Mean: {text_embeddings.mean():.4f}, Std: {text_embeddings.std():.4f}")
else:
    print("No transcript found")

## 2.4 Combine into unified feature dict per patient

This is the format expected by `TrimodalDataset`.

In [ ]:
def build_patient_feature_dict(patient_dir: Path, pid: str) -> dict:
    """Build unified feature dict for one patient."""
    result = {}
    
    # Audio: concatenate MFCC + eGeMAPS
    audio_dir = patient_dir / "audio_features"
    mfcc_path = [f for f in audio_dir.iterdir() if 'mfcc' in f.name.lower() and f.suffix == '.csv']
    egemaps_path = [f for f in audio_dir.iterdir() if 'egemap' in f.name.lower() and f.suffix == '.csv']
    if mfcc_path and egemaps_path:
        mfcc_df = pd.read_csv(mfcc_path[0])
        egemaps_df = pd.read_csv(egemaps_path[0])
        exclude = {'name', 'frameTime', 'file'}
        mfcc_arr = mfcc_df[[c for c in mfcc_df.columns if c not in exclude]].values.astype(np.float32)
        egemaps_arr = egemaps_df[[c for c in egemaps_df.columns if c not in exclude]].values.astype(np.float32)
        min_len = min(len(mfcc_arr), len(egemaps_arr))
        result['audio'] = np.concatenate([mfcc_arr[:min_len], egemaps_arr[:min_len]], axis=1)
    
    # Video
    video_features = VideoFeatureExtractor().extract_from_folder(patient_dir)
    result['video_openface'] = video_features.get('openface')
    result['video_cnn'] = video_features.get('cnn_embed')
    
    # Text
    transcript = patient_dir / "transcript.csv"
    if transcript.exists():
        result['text'] = TextExtractor().load_from_csv(transcript)
    
    return result

features = build_patient_feature_dict(PATIENTS_DIR / sample_pid, sample_pid)

print("Unified feature dict:")
for k, v in features.items():
    if v is not None:
        print(f"  {k}: {v.shape}")
    else:
        print(f"  {k}: None")

## 2.5 Batch feature extraction across all patients

In [ ]:
from tqdm.notebook import tqdm

all_features = {}
failed = []

for patient_dir in tqdm(sorted([d for d in PATIENTS_DIR.iterdir() if d.is_dir()])):
    pid = patient_dir.name
    try:
        feats = build_patient_feature_dict(patient_dir, pid)
        all_features[pid] = feats
    except Exception as e:
        failed.append((pid, str(e)))

print(f"\nSuccess: {len(all_features)} patients")
print(f"Failed: {len(failed)} patients")
if failed:
    for pid, err in failed[:5]:
        print(f"  {pid}: {err}")